# Wallpaper Group Hierarchy Vector Builder

## 1. Setup

In [2]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import h5py
import json
import pandas as pd

from tqdm import tqdm
import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import DataLoader

from dl_utils.utils.dataset import viz_dataloader, split_train_valid, hdf5_dataset
from dl_utils.training.build_model import resnet50_, xcit_small
from dl_utils.training.trainer import Trainer, accuracy
from dl_utils.packed_functions import benchmark_task
from dl_utils.utils.utils import viz_h5_structure


/mnt/scratch/home/yichen/anaconda3/envs/symmetry/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
symmetry_classes = [
    'p1', 'p2', 'pm', 'pg', 'cm', 'pmm', 'pmg', 'pgg',
    'cmm', 'p4', 'p4m', 'p4g', 'p3', 'p3m1', 'p31m', 'p6', 'p6m',
]

symmetry_dict = {name: idx for idx, name in enumerate(symmetry_classes)}

hierarchy_feature_labels = [
    'lattice_oblique', 'lattice_rect_prim', 'lattice_rect_centered', 'lattice_square',
    'lattice_hex', 'rot_ord_1', 'rot_ord_2', 'rot_ord_3', 'rot_ord_4', 'rot_ord_6',
    'mirror_cnt_0', 'mirror_cnt_1', 'mirror_cnt_2', 'mirror_cnt_3',
    'glide_cnt_0', 'glide_cnt_1', 'glide_cnt_2', 'glide_cnt_3',
    'has_axis_mirror', 'has_diagonal_mirror', 'arrangement_p3m1', 'arrangement_p31m',
]

symmetry_vectors = {
    'p1': np.array([1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0], dtype=np.uint8),
    'p2': np.array([1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0], dtype=np.uint8),
    'pm': np.array([0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0], dtype=np.uint8),
    'pg': np.array([0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0], dtype=np.uint8),
    'cm': np.array([0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0], dtype=np.uint8),
    'pmm': np.array([0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0], dtype=np.uint8),
    'pmg': np.array([0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0], dtype=np.uint8),
    'pgg': np.array([0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], dtype=np.uint8),
    'cmm': np.array([0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0], dtype=np.uint8),
    'p4': np.array([0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0], dtype=np.uint8),
    'p4m': np.array([0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0], dtype=np.uint8),
    'p4g': np.array([0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0], dtype=np.uint8),
    'p3': np.array([0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0], dtype=np.uint8),
    'p3m1': np.array([0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0], dtype=np.uint8),
    'p31m': np.array([0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1], dtype=np.uint8),
    'p6': np.array([0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0], dtype=np.uint8),
    'p6m': np.array([0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0], dtype=np.uint8),
}

### Symmetry Vector Feature Glossary
Each symmetry vector entry is a binary indicator derived from wallpaper-group taxonomy. The values describe lattice type, rotational orders, and the presence/arrangement of reflection or glide operations.

- `lattice_oblique`: Oblique (parallelogram) Bravais lattice without orthogonal basis vectors.
- `lattice_rect_prim`: Primitive rectangular lattice (orthogonal basis, no centering).
- `lattice_rect_centered`: Centered rectangular lattice (rhombic, rectangular net with a centered point).
- `lattice_square`: Square lattice with equal orthogonal basis vectors.
- `lattice_hex`: Hexagonal lattice built from 60° basis vectors.
- `rot_ord_1`: Only 1-fold (identity) rotational symmetry beyond translations.
- `rot_ord_2`: Contains at least one 2-fold rotation center.
- `rot_ord_3`: Contains at least one 3-fold rotation center.
- `rot_ord_4`: Contains at least one 4-fold rotation center.
- `rot_ord_6`: Contains at least one 6-fold rotation center.
- `mirror_cnt_0`: No mirror-reflection lines are present.
- `mirror_cnt_1`: One family of mutually parallel mirror lines.
- `mirror_cnt_2`: Two families of mirror lines (typically orthogonal).
- `mirror_cnt_3`: Three or more mirror families (e.g., triangular / hexagonal arrangements).
- `glide_cnt_0`: No glide reflections.
- `glide_cnt_1`: One family of glide reflections.
- `glide_cnt_2`: Two distinct glide families.
- `glide_cnt_3`: Three or more glide families.
- `has_axis_mirror`: Mirrors aligned with the principal lattice axes.
- `has_diagonal_mirror`: Mirrors aligned along lattice diagonals (45°/60°).
- `arrangement_p3m1`: Triangular-lattice mirror arrangement matching the p3m1 convention.
- `arrangement_p31m`: Triangular-lattice mirror arrangement matching the p31m convention.


## 2. Inspect Input Metadata

In [4]:
input_file = Path("../../datasets/imagenet_v5_rot_10m_fix_vector.h5")
output_file = Path("../../datasets/imagenet_v5_rot_10m_fix_vector-symmetry_vector.h5")

with h5py.File(input_file, "r") as h5:
    viz_h5_structure(h5)


'Group': imagenet
  'Dataset': data; Shape: (10173208, 256, 256, 3); dtype: uint8
  'Dataset': labels; Shape: (10173208,); dtype: uint8
  'Dataset': primitive_uc_vector_a; Shape: (10173208, 2); dtype: int32
  'Dataset': primitive_uc_vector_b; Shape: (10173208, 2); dtype: int32
  'Dataset': rotation_angle; Shape: (10173208,); dtype: uint8
  'Dataset': shape; Shape: (10173208,); dtype: uint8
  'Dataset': translation_start_point; Shape: (10173208, 2); dtype: int32
  'Dataset': translation_uc_vector_a; Shape: (10173208, 2); dtype: int32
  'Dataset': translation_uc_vector_b; Shape: (10173208, 2); dtype: int32


## 3. Build Class Hierarchy Vectors

In [4]:
# 3) Copy metadata without raw images and build symmetry vectors
index_to_group = {idx: name for name, idx in symmetry_dict.items()}

with h5py.File(input_file, 'r') as src_h5, h5py.File(output_file, 'w') as dst_h5:
    # Preserve root-level attributes
    for key, value in src_h5.attrs.items():
        dst_h5.attrs[key] = value

    src_grp = src_h5['imagenet']
    dst_grp = dst_h5.create_group('imagenet')

    # Carry over group-level attributes
    for key, value in src_grp.attrs.items():
        dst_grp.attrs[key] = value

    # Copy everything except the heavy raw image tensor
    for name in src_grp.keys():
        if name == 'data':
            continue
        src_grp.copy(name, dst_grp, name=name)

    labels = dst_grp['labels'][:]
    symmetry_array = np.stack([
        symmetry_vectors[index_to_group[int(lbl)]]
        for lbl in labels
    ]).astype(np.uint8)

    if 'symmetry_vector' in dst_grp:
        del dst_grp['symmetry_vector']
    sym_dset = dst_grp.create_dataset('symmetry_vector', data=symmetry_array, dtype=np.uint8)
    sym_dset.attrs['description'] = (
        'Wallpaper-group symmetry vector (22 dims): ' + ', '.join(hierarchy_feature_labels)
    )

    dst_h5.attrs['hierarchy_feature_labels'] = json.dumps(hierarchy_feature_labels)
    dst_h5.attrs['symmetry_vectors_lookup'] = json.dumps(
        {k: v.astype(int).tolist() for k, v in symmetry_vectors.items()}
    )
    dst_h5.attrs['symmetry_dict'] = json.dumps(symmetry_dict)
    dst_h5.attrs['symmetry_vector_comment'] = (
        'symmetry_vector stored under imagenet uses 22-dim hierarchy features.'
    )
    dst_h5.attrs['copied_without_data'] = True

print(f"Saved symmetry vectors to {output_file}")



Saved symmetry vectors to ../../datasets/imagenet_v5_rot_10m_fix_vector-symmetry_vector.h5


## 4. Normalize Geometry (optional)

In [5]:
import h5py
import numpy as np
from sklearn.preprocessing import MinMaxScaler

with h5py.File(input_file, "r") as h5:
    f = h5['imagenet']
    primitive_uc_vector_a = f['primitive_uc_vector_a'][:]
    primitive_uc_vector_b = f['primitive_uc_vector_b'][:]
    translation_start_point = f['translation_start_point'][:]

vectors = np.hstack([
    primitive_uc_vector_a,
    primitive_uc_vector_b,
    translation_start_point,
]).astype(np.float32)

scaler = MinMaxScaler()
vectors_norm = scaler.fit_transform(vectors)

norm_a, norm_b, norm_trans = np.split(vectors_norm, 3, axis=1)

with h5py.File(output_file, 'a') as dst_h5:
    grp = dst_h5['imagenet']
    for name, data in {
        'normalized_primitive_uc_vector_a': norm_a,
        'normalized_primitive_uc_vector_b': norm_b,
        'normalized_translation_start_point': norm_trans,
    }.items():
        if name in grp:
            del grp[name]
        ds = grp.create_dataset(name, data=data.astype(np.float32))
        ds.attrs['description'] = name.replace('_', ' ') + ' (min-max scaled).'

    similarity = np.hstack([norm_a, norm_b, norm_trans]).astype(np.float32)
    if 'similarity_vector' in grp:
        del grp['similarity_vector']
    ds = grp.create_dataset('similarity_vector', data=similarity)
    ds.attrs['description'] = 'Concatenated normalized geometry: prim_a(2), prim_b(2), translation_start(2).'

    dst_h5.attrs['geometry_vector_comment'] = 'similarity_vector stores normalized geometry features.'

print('Normalized geometry written to hierarchy file.')


Normalized geometry written to hierarchy file.


## 5. Inspect Enriched File

In [6]:
with h5py.File(output_file, "r") as h5:
    viz_h5_structure(h5)


'Group': imagenet
  'Dataset': labels; Shape: (10173208,); dtype: uint8
  'Dataset': normalized_primitive_uc_vector_a; Shape: (10173208, 2); dtype: float32
  'Dataset': normalized_primitive_uc_vector_b; Shape: (10173208, 2); dtype: float32
  'Dataset': normalized_translation_start_point; Shape: (10173208, 2); dtype: float32
  'Dataset': primitive_uc_vector_a; Shape: (10173208, 2); dtype: int32
  'Dataset': primitive_uc_vector_b; Shape: (10173208, 2); dtype: int32
  'Dataset': rotation_angle; Shape: (10173208,); dtype: uint8
  'Dataset': shape; Shape: (10173208,); dtype: uint8
  'Dataset': similarity_vector; Shape: (10173208, 6); dtype: float32
  'Dataset': symmetry_vector; Shape: (10173208, 22); dtype: uint8
  'Dataset': translation_start_point; Shape: (10173208, 2); dtype: int32
  'Dataset': translation_uc_vector_a; Shape: (10173208, 2); dtype: int32
  'Dataset': translation_uc_vector_b; Shape: (10173208, 2); dtype: int32


## 6. Inspect Stored Attributes

In [9]:
with h5py.File(output_file, "r") as h5:
    vecs = h5["imagenet"]["symmetry_vector"]     # dataset
    print(vecs.shape, vecs[:2])

    print(h5.attrs["symmetry_vectors_lookup"])   # lookup table JSON
    print(h5.attrs["symmetry_vector_comment"])   # description


(10173208, 22) [[0 0 0 0 1 0 0 0 0 1 1 0 0 0 1 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 1 0 0 0 0 1 0 0 0 1 0 0 1 0 0 0]]
{"p1": [1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0], "p2": [1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0], "pm": [0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0], "pg": [0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0], "cm": [0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0], "pmm": [0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0], "pmg": [0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0], "pgg": [0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], "cmm": [0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0], "p4": [0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0], "p4m": [0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0], "p4g": [0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0,

In [10]:
with h5py.File(output_file, "r") as h5:
    keys = [
        'symmetry_dict',
        'hierarchy_feature_labels',
        'symmetry_vectors_lookup',
        'geometry_vector_comment',
        'symmetry_vector_comment',
    ]
    for key in keys:
        if key not in h5.attrs:
            print(f"{key}: <missing>")
            continue
        raw = h5.attrs[key]
        if isinstance(raw, str) and raw and raw[0] in "{[":
            try:
                raw = json.loads(raw)
            except json.JSONDecodeError:
                pass
        print(f"{key}: {raw}")




symmetry_dict: {'p1': 0, 'p2': 1, 'pm': 2, 'pg': 3, 'cm': 4, 'pmm': 5, 'pmg': 6, 'pgg': 7, 'cmm': 8, 'p4': 9, 'p4m': 10, 'p4g': 11, 'p3': 12, 'p3m1': 13, 'p31m': 14, 'p6': 15, 'p6m': 16}
hierarchy_feature_labels: ['lattice_oblique', 'lattice_rect_prim', 'lattice_rect_centered', 'lattice_square', 'lattice_hex', 'rot_ord_1', 'rot_ord_2', 'rot_ord_3', 'rot_ord_4', 'rot_ord_6', 'mirror_cnt_0', 'mirror_cnt_1', 'mirror_cnt_2', 'mirror_cnt_3', 'glide_cnt_0', 'glide_cnt_1', 'glide_cnt_2', 'glide_cnt_3', 'has_axis_mirror', 'has_diagonal_mirror', 'arrangement_p3m1', 'arrangement_p31m']
symmetry_vectors_lookup: {'p1': [1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0], 'p2': [1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0], 'pm': [0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0], 'pg': [0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0], 'cm': [0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0], 'pmm': [0, 1, 0, 0, 0

## 7. Sample Record

In [6]:
with h5py.File(output_file, 'r') as h5:
    grp = h5['imagenet']
    for idx in range(0, 100):
        # idx = 0

        symmetry_lookup = {int(v): k for k, v in json.loads(h5.attrs['symmetry_dict']).items()}
        feature_labels = json.loads(h5.attrs['hierarchy_feature_labels'])

        label_id = int(grp['labels'][idx])
        class_name = symmetry_lookup[label_id]
    
        if  class_name == 'p1':
            
            hierarchy_vec = grp['symmetry_vector'][idx]
            geometry_vec = grp['similarity_vector'][idx]

            print(f"Sample {idx}:")
            print(f"  Wallpaper group : {class_name} ({label_id})")
            print("  Hierarchy vector:")
            for label, value in zip(feature_labels, hierarchy_vec):
                print(f"    {label}: {int(value)}")
            print("  Geometry vector:", geometry_vec)

Sample 48:
  Wallpaper group : p1 (0)
  Hierarchy vector:
    lattice_oblique: 1
    lattice_rect_prim: 0
    lattice_rect_centered: 0
    lattice_square: 0
    lattice_hex: 0
    rot_ord_1: 1
    rot_ord_2: 0
    rot_ord_3: 0
    rot_ord_4: 0
    rot_ord_6: 0
    mirror_cnt_0: 1
    mirror_cnt_1: 0
    mirror_cnt_2: 0
    mirror_cnt_3: 0
    glide_cnt_0: 1
    glide_cnt_1: 0
    glide_cnt_2: 0
    glide_cnt_3: 0
    has_axis_mirror: 0
    has_diagonal_mirror: 0
    arrangement_p3m1: 0
    arrangement_p31m: 0
  Geometry vector: [0.6492147  0.5803109  0.5792683  0.3393939  0.40053052 0.43467337]
Sample 51:
  Wallpaper group : p1 (0)
  Hierarchy vector:
    lattice_oblique: 1
    lattice_rect_prim: 0
    lattice_rect_centered: 0
    lattice_square: 0
    lattice_hex: 0
    rot_ord_1: 1
    rot_ord_2: 0
    rot_ord_3: 0
    rot_ord_4: 0
    rot_ord_6: 0
    mirror_cnt_0: 1
    mirror_cnt_1: 0
    mirror_cnt_2: 0
    mirror_cnt_3: 0
    glide_cnt_0: 1
    glide_cnt_1: 0
    glide_cnt_2: 0


## 8. Tabular Preview

In [ ]:
import pandas as pd

with h5py.File(output_file, 'r') as h5:
    grp = h5['imagenet']
    n = min(20, len(grp['labels']))
    symmetry_lookup = {int(v): k for k, v in json.loads(h5.attrs['symmetry_dict']).items()}
    feature_labels = json.loads(h5.attrs['hierarchy_feature_labels'])

    df = pd.DataFrame({
        'GroupId': grp['labels'][:n],
        'Group': [symmetry_lookup[int(label)] for label in grp['labels'][:n]],
    })
    hierarchy_df = pd.DataFrame(
        grp['symmetry_vector'][:n],
        columns=feature_labels,
    ).astype(int)

    df = pd.concat([df, hierarchy_df], axis=1)
df

,GroupId,Group,lattice_oblique,lattice_rect_prim,lattice_rect_centered,lattice_square,lattice_hex,rot_ord_1,rot_ord_2,rot_ord_3,...,mirror_cnt_2,mirror_cnt_3,glide_cnt_0,glide_cnt_1,glide_cnt_2,glide_cnt_3,has_axis_mirror,has_diagonal_mirror,arrangement_p3m1,arrangement_p31m
0,15,p6,0,0,0,0,1,0,0,0,...,0,0,1,0,0,0,0,0,0,0
1,6,pmg,0,1,0,0,0,0,1,0,...,0,0,0,1,0,0,1,0,0,0
2,5,pmm,0,1,0,0,0,0,1,0,...,1,0,1,0,0,0,1,0,0,0
3,1,p2,1,0,0,0,0,0,1,0,...,0,0,1,0,0,0,0,0,0,0
4,4,cm,0,0,1,0,0,1,0,0,...,0,0,0,1,0,0,1,0,0,0
5,9,p4,0,0,0,1,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
6,2,pm,0,1,0,0,0,1,0,0,...,0,0,1,0,0,0,1,0,0,0
7,14,p31m,0,0,0,0,1,0,0,1,...,0,1,1,0,0,0,1,0,0,1
8,12,p3,0,0,0,0,1,0,0,1,...,0,0,1,0,0,0,0,0,0,0
9,6,pmg,0,1,0,0,0,0,1,0,...,0,0,0,1,0,0,1,0,0,0


## 9. Verify Hierarchy Vector

In [13]:
import numpy as np

with h5py.File(output_file, 'r') as h5:
    grp = h5['imagenet']
    symmetry_dict = json.loads(h5.attrs['symmetry_dict'])
    hierarchy_lookup = {k: np.array(v, dtype=np.uint8) for k, v in json.loads(h5.attrs['symmetry_vectors_lookup']).items()}

    labels = grp['labels'][:]
    stored = grp['symmetry_vector'][:]

    index_to_group = {int(v): k for k, v in symmetry_dict.items()}

    recomputed = np.stack([
        hierarchy_lookup[index_to_group[int(lbl)]]
        for lbl in labels
    ]).astype(np.uint8)

    diffs = np.argwhere(recomputed != stored)
    if diffs.size == 0:
        print('Verification passed: stored vectors match lookup table.')
    else:
        print('Verification FAILED: mismatches at indices:', diffs[:10])



Verification passed: stored vectors match lookup table.
